# Stage 1.5: geometry + rooted Weisfeiler–Lehman colours

Adapt the fixed 1-WL refinement from `notebooks/05_Shallow_Learning/code03_solution.ipynb` to per-face classification. Every face starts with colour `"0"`. At each round, hash its previous colour together with the sorted multiset of neighboring colours, using the tutorial's BLAKE2b function. Retain **each face's own colour history**, rather than the tutorial's graph-wide histogram. Compute each solid independently on the simple undirected graph; remove self-loops and collapse parallel edges.

Choose frozen CNN or joint CNN + classifier. WL topology preprocessing stays fixed in both modes. There is no learned graph message passing and no graph-level kernel/SVM in this notebook. Class labels never initialize WL colours.

Default `wl_iterations=3` includes rounds 0–3. The same hash identifies the same rooted pattern across solids. Fit a sorted vocabulary on training colours only; unseen validation colours contribute zero. No random sign augmentation is needed.


In [ ]:
import os
os.environ["DGLBACKEND"] = "pytorch"

import csv
import hashlib
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from uvnet_project.checkpointing import seed_everything, save_training_checkpoint, load_training_checkpoint
from uvnet_project.face_cnn import FaceCNN
from uvnet_project.wl import wl_node_colors, fit_wl_vocabulary, wl_color_maps, encode_wl_colors
from uvnet_project.geometry_wl import FaceWLBatch, WLClassifier
from uvnet_project.load_mfcad import DATA_DIR, _load_sample, _load_splits
from uvnet_project.evaluation import evaluate


## Reproducibility and resume

Fresh runs reset Python, NumPy and Torch CPU/CUDA seeds. Each epoch exports model weights and a full atomic checkpoint with optimizer, optional AMP scaler, seed/RNG states, histories, settings and the training vocabulary. Set `resume_checkpoint` to a completed epoch and `num_epochs` to the total target, then rerun preparation and training cells. Keep the same caches, dataset order, software and GPU for reproducible replay. Mid-epoch interruptions resume from the previous completed epoch. Existing earlier-format checkpoints without RNG states cannot reproduce an exact trajectory.

Use a fresh kernel before deterministic CUDA settings. No training is run during notebook preparation.


## Settings and baseline CNN

Use `training_mode="frozen"` or `"joint"` and choose `wl_iterations` (default 3, matching the tutorial). Start both from the completed baseline CNN. Each round count and mode has a separate results folder. Validation is the official validation split; test is not loaded. Matching run files are overwritten. Training defaults to 20 epochs and 128 solids per batch.


In [ ]:
seed = 0
deterministic = True
# Set a completed WL checkpoint to resume, then rerun preparation cells.
# resume_checkpoint = DATA_DIR.parent.parent / "results/wl3_frozen_run_01/epoch_010/checkpoint.pt"
resume_checkpoint = None
seed_everything(seed, deterministic=deterministic)

project_dir = DATA_DIR.parent.parent
baseline_checkpoint = project_dir / "results/baseline_run_01/epoch_020/checkpoint.pt"
training_mode = "frozen"  # "frozen" or "joint"
wl_iterations = 3  # tutorial default: include colours at rounds 0, 1, 2, 3
num_epochs = 20
batch_size = 128  # solids per batch
learning_rate = 0.001
run_dir = project_dir / "results" / f"wl{wl_iterations}_{training_mode}_run_01"
cache_dir = project_dir / "data/feature_cache"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if training_mode not in ("frozen", "joint") or wl_iterations < 0:
    raise ValueError("Choose frozen/joint and a nonnegative WL iteration count.")
if min(num_epochs, batch_size) < 1:
    raise ValueError("Epoch count and batch size must be positive.")
if not baseline_checkpoint.is_file():
    raise FileNotFoundError(f"Copy the completed baseline checkpoint here first: {baseline_checkpoint}")
checkpoint = torch.load(baseline_checkpoint, map_location="cpu", weights_only=True)
config = checkpoint["config"]
embedding_dim, hidden_dim = config["embedding_dim"], config["hidden_dim"]
num_classes, in_channels = config["num_classes"], config["in_channels"]
baseline_hash = hashlib.sha256(baseline_checkpoint.read_bytes()).hexdigest()
cnn_state = {key.removeprefix("0."): value for key, value in checkpoint["model_state_dict"].items()
             if key.startswith("0.")}
face_cnn = FaceCNN(embedding_dim=embedding_dim, in_channels=in_channels)
face_cnn.load_state_dict(cnn_state, strict=True)
face_cnn.requires_grad_(False)
face_cnn = face_cnn.to(device).eval()
print("Device:", device, "Mode:", training_mode, "WL rounds:", wl_iterations)


## Cache WL colours, fit the training vocabulary, and prepare inputs

The colour cache is independent of CNN weights. Validation may generate unseen hashes, but it cannot add vocabulary columns. Known categories use distinct IDs for each round; the first ID in each round is reserved for unknowns. IDs are categorical lookup indices, not numerical measurements. Vocabulary ordering is deterministic and its hash is checked on resume.

Frozen mode caches baseline geometry embeddings. Joint mode caches raw face grids so the CNN can adapt. Inputs and colour histories preserve split and face order. Delete caches or increment `cache_version` if dataset files or the algorithm change.


In [ ]:
def metadata_key(metadata):
    return hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]


@torch.no_grad()
def prepare_colors(split):
    sample_ids = _load_splits()[split]
    metadata = dict(cache_version=1, sample_ids=sample_ids, wl_iterations=wl_iterations,
                    encoding="simple_undirected_uniform_0_blake2b16_node_history")
    path = cache_dir / f"wl_colors_{split}_{metadata_key(metadata)}.pt"
    if path.is_file():
        cached = torch.load(path, map_location="cpu", weights_only=True)
        if cached["metadata"] != metadata:
            raise ValueError("WL colour-cache metadata mismatch.")
        print("Using colour cache:", path)
        return cached["histories"], sample_ids
    histories = []
    for sample_id in tqdm(sample_ids, desc=f"WL colours {split}"):
        solid = _load_sample(sample_id)
        histories.append(wl_node_colors(solid["graph"], wl_iterations))
    cache_dir.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".pt.tmp")
    torch.save(dict(metadata=metadata, histories=histories), temporary)
    os.replace(temporary, path)
    return histories, sample_ids


train_histories, train_ids = prepare_colors("train")
# Validation colours never participate in fitting the vocabulary.
vocabulary = fit_wl_vocabulary(train_histories, wl_iterations)
vocabulary_hash = hashlib.sha256(json.dumps(vocabulary, sort_keys=True).encode()).hexdigest()
color_maps = wl_color_maps(vocabulary)
vocabulary_path = cache_dir / f"wl_vocabulary_{vocabulary_hash[:16]}.json"
vocabulary_path.write_text(json.dumps(vocabulary, indent=1) + "\n")
validation_histories, validation_ids = prepare_colors("validation")
print("Known training colours per round:", [len(values) for values in vocabulary["colors"]])
print("Total categorical input columns including unknowns:", vocabulary["size"])


@torch.no_grad()
def prepare_inputs(split, histories, sample_ids, encode_batch_size=128):
    metadata = dict(cache_version=1, baseline_hash=baseline_hash, vocabulary_hash=vocabulary_hash,
                    wl_iterations=wl_iterations, training_mode=training_mode,
                    sample_ids=sample_ids, embedding_dim=embedding_dim, in_channels=in_channels)
    path = cache_dir / f"wl_inputs_{split}_{metadata_key(metadata)}.pt"
    if path.is_file():
        cached = torch.load(path, map_location="cpu", weights_only=True)
        if cached["metadata"] != metadata:
            raise ValueError("WL input-cache metadata mismatch.")
        print("Using input cache:", path)
        return cached["samples"]
    if resume_checkpoint is not None:
        raise FileNotFoundError(f"Retain the original WL input cache for resume: {path}")
    face_cnn.eval()
    samples = []
    for start in tqdm(range(0, len(sample_ids), encode_batch_size), desc=f"Cache WL inputs {split}"):
        solids = [_load_sample(sample_id) for sample_id in sample_ids[start:start + encode_batch_size]]
        embeddings = None
        if training_mode == "frozen":
            sizes = [solid["graph"].num_nodes() for solid in solids]
            grids = torch.cat([solid["graph"].ndata["x"] for solid in solids])
            grids = grids.permute(0, 3, 1, 2).contiguous().float().to(device)
            embeddings = face_cnn(grids).cpu().split(sizes)
        for offset, solid in enumerate(solids):
            colors = encode_wl_colors(histories[start + offset], vocabulary, color_maps)
            labels = torch.tensor([face["segment"]["index"] for face in solid["faces"]], dtype=torch.long)
            if training_mode == "frozen":
                geometry = embeddings[offset]
            else:
                geometry = solid["graph"].ndata["x"].permute(0, 3, 1, 2).contiguous().float()
            assert len(colors) == len(labels)
            samples.append((geometry, colors, labels))
    temporary = path.with_suffix(".pt.tmp")
    torch.save(dict(metadata=metadata, samples=samples), temporary)
    os.replace(temporary, path)
    return samples


train_samples = prepare_inputs("train", train_histories, train_ids)
validation_samples = prepare_inputs("validation", validation_histories, validation_ids)


## Build the classifier

Conceptually concatenate geometry with one-hot vectors for the face's WL colours across all rounds, then apply `Linear -> ReLU -> Linear`. To avoid enormous dense one-hot matrices, sum the first layer's selected colour-weight rows using an embedding lookup and add the geometry projection. This is mathematically the same classifier, not an extra WL network. Unknown-colour contributions are masked to zero.

Frozen mode trains only these classifier weights. Joint mode also trains the CNN and updates its batch-normalization statistics. The vocabulary and colour assignments never change during training. Print the vocabulary size and parameter count to assess the increased classifier capacity.


In [ ]:
def build_model():
    head = WLClassifier(embedding_dim, hidden_dim, vocabulary, num_classes)
    if training_mode == "joint":
        # Keep initialization of classifier/shuffling identical between modes.
        with torch.random.fork_rng(devices=[]):
            encoder = FaceCNN(embedding_dim=embedding_dim, in_channels=in_channels)
        encoder.load_state_dict(cnn_state, strict=True)
        encoder.requires_grad_(True)
        head.face_cnn = encoder
    return head.to(device)


seed_everything(seed, deterministic=deterministic)
model = build_model()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Batches and unseen-colour diagnostics

Batches retain categorical IDs as integers; the classifier performs one-hot-equivalent projection. No scalar normalization of arbitrary colour IDs is used. Display unseen-colour rates separately for training and validation at each WL round. Unknown colours have a zero topology contribution for that round; geometry and known rounds remain available.


In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        geometry, colors, labels = self.samples[index]
        return (geometry, colors), labels


def collate_features(batch):
    inputs, labels = zip(*batch)
    geometry, colors = zip(*inputs)
    return FaceWLBatch(torch.cat(geometry), torch.cat(colors)), torch.cat(labels)


loader_options = dict(batch_size=batch_size, collate_fn=collate_features,
                      num_workers=0, pin_memory=device.type == "cuda")
train_loader = DataLoader(FeatureDataset(train_samples), shuffle=True, **loader_options)
train_evaluation_loader = DataLoader(FeatureDataset(train_samples), shuffle=False, **loader_options)
validation_loader = DataLoader(FeatureDataset(validation_samples), shuffle=False, **loader_options)
inputs, labels = next(iter(validation_loader))
model.eval()
with torch.no_grad():
    logits = model(inputs.to(device))
assert logits.shape == (len(labels), num_classes)
assert torch.isfinite(logits).all()
unknown = torch.tensor(vocabulary["unknown_indices"])
for split, samples in (("train", train_samples), ("validation", validation_samples)):
    tokens = torch.cat([sample[1] for sample in samples])
    rates = (tokens == unknown).float().mean(0)
    print(split, "unseen-colour fraction by round:", [round(value, 4) for value in rates.tolist()])
print("Logits:", tuple(logits.shape))


## Train the chosen WL variant — run when ready

Record batch training loss, evaluate validation each epoch, and save weights plus resumable checkpoints every epoch. Frozen mode uses float32; joint CUDA mode uses AMP with saved scaler state. Training metrics accumulate during updates; validation uses evaluation mode. Prediction probabilities are saved only at the final epoch. Loss plotting is separate and runs after training finishes.


In [ ]:
seed_everything(seed, deterministic=deterministic)
if resume_checkpoint is None:
    model = build_model()
use_amp = training_mode == "joint" and device.type == "cuda"
if use_amp:
    model = model.to(memory_format=torch.channels_last)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
loss_fn = nn.CrossEntropyLoss()
history, batch_history = [], []
global_step = 0

run_config = dict(stage="wl", seed=seed, deterministic=deterministic,
                  embedding_dim=embedding_dim, wl_iterations=wl_iterations,
                  vocabulary_size=vocabulary["size"], vocabulary_hash=vocabulary_hash,
                  hidden_dim=hidden_dim, num_classes=num_classes,
                  learning_rate=learning_rate, batch_size=batch_size,
                  baseline_hash=baseline_hash, train_sample_ids=train_ids,
                  validation_sample_ids=validation_ids, training_mode=training_mode,
                  mixed_precision=use_amp)
start_epoch = 1
if resume_checkpoint is not None:
    resume_info = torch.load(resume_checkpoint, map_location="cpu", weights_only=True)
    saved_config = resume_info["config"]
    # Resume must use the same WL rounds, vocabulary and training mode.
    if saved_config.get("training_mode", "frozen") != training_mode:
        raise ValueError("Cannot resume a different training mode.")
    if saved_config.get("mixed_precision", False) != use_amp:
        raise ValueError("Resume mixed-precision setting does not match.")
    expected_config = {key: value for key, value in run_config.items()
                       if key not in ("training_mode", "mixed_precision") or key in saved_config}
    restored = load_training_checkpoint(
        resume_checkpoint, model, optimizer, expected_config=expected_config, scaler=scaler if use_amp else None,
    )
    start_epoch = restored["epoch"] + 1
    history = restored["history"]
    batch_history = restored["batch_history"]
    global_step = batch_history[-1]["step"] if batch_history else 0
    if num_epochs < start_epoch:
        raise ValueError("Set num_epochs above the completed checkpoint epoch to continue training.")
    print(f"Resuming at epoch {start_epoch}; target is epoch {num_epochs}.")
run_dir.mkdir(parents=True, exist_ok=True)


@torch.no_grad()
def validation_metrics():
    model.eval()
    loss_total = torch.zeros((), device=device)
    correct_total = torch.zeros((), dtype=torch.long, device=device)
    faces = 0
    for features, labels in validation_loader:
        features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(features)
            loss = loss_fn(logits, labels)
        loss_total += loss * labels.numel()
        correct_total += (logits.argmax(1) == labels).sum()
        faces += labels.numel()
    return loss_total.item() / faces, correct_total.item() / faces


for epoch in tqdm(range(start_epoch, num_epochs + 1), desc="Epochs"):
    model.train()
    losses, correct, sizes = [], [], []
    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch}: training", leave=False):
        features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(features)
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.detach())
        correct.append((logits.detach().argmax(1) == labels).sum())
        sizes.append(labels.numel())

    loss_values = torch.stack(losses).cpu().tolist()
    train_loss = sum(loss * size for loss, size in zip(loss_values, sizes)) / sum(sizes)
    train_accuracy = torch.stack(correct).sum().item() / sum(sizes)
    for batch_index, (batch_loss, count) in enumerate(zip(loss_values, sizes), 1):
        global_step += 1
        batch_history.append(dict(step=global_step, epoch=epoch, batch=batch_index,
                                  loss=batch_loss, faces=count))
    val_loss, val_accuracy = validation_metrics()
    row = dict(epoch=epoch, step=global_step, train_loss=train_loss,
               train_accuracy=train_accuracy, validation_loss=val_loss,
               validation_accuracy=val_accuracy)
    history.append(row)
    checkpoint_dir = run_dir / f"epoch_{epoch:03d}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    save_training_checkpoint(
        checkpoint_dir / "checkpoint.pt", model, optimizer, epoch=epoch, seed=seed,
        config={**run_config, "num_epochs": num_epochs, "checkpoint_every": 1},
        history=history, batch_history=batch_history, scaler=scaler if use_amp else None,
        extra=dict(wl_vocabulary=vocabulary, baseline_checkpoint=str(baseline_checkpoint), baseline_hash=baseline_hash, face_cnn_state_dict=(model.face_cnn if training_mode == "joint" else face_cnn).state_dict(), train_sample_ids=train_ids, validation_sample_ids=validation_ids),
    )
    with (run_dir / "history.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(history)
    tqdm.write(f"Epoch {epoch:02d}/{num_epochs} | train loss {train_loss:.4f}, "
               f"accuracy {train_accuracy:.2%} | validation loss {val_loss:.4f}, "
               f"accuracy {val_accuracy:.2%}")
    with (run_dir / "batch_loss.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(batch_history[0]))
        writer.writeheader()
        writer.writerows(batch_history)

final_train_results = evaluate(model, train_evaluation_loader,
                               checkpoint_dir / "train_predictions.pt", device=device)
final_validation_results = evaluate(model, validation_loader,
                                    checkpoint_dir / "validation_predictions.pt", device=device)
print("Final predictions saved to:", checkpoint_dir)


## Loss plot after training
Run after the training cell completes.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

batch_log = pd.DataFrame(batch_history)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_log["step"], batch_log["loss"], alpha=0.25, label="Training batch loss")
ax.plot(batch_log["step"], batch_log["loss"].rolling(20, min_periods=1).mean(),
        label="Training loss (20-batch moving average)")
ax.plot([row["step"] for row in history],
        [row["validation_loss"] for row in history], marker="o", label="Validation loss")
ax.set_xlabel("Training step (batch)")
ax.set_ylabel("Mean cross-entropy loss per face")
ax.set_title(f"{training_mode.capitalize()} CNN + WL (T={wl_iterations})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir / "loss.png", dpi=150)
plt.show()


## Accuracy and IoU: baseline + WL variants

Read final cached training/validation predictions for the baseline and completed WL runs only. Show checkpoint epoch, loss, accuracy, unweighted mean IoU (including stock), and per-class IoU. Missing runs are skipped. No inference or training.


In [ ]:
# Compare cached final predictions across all completed variants.
from pathlib import Path
import json
import pandas as pd
import torch
from IPython.display import display
from uvnet_project.load_mfcad import DATA_DIR

project_dir = DATA_DIR.parent.parent
import re
variant_dirs = {"Geometry only": project_dir / "results/baseline_run_01"}
for folder in sorted((project_dir / "results").glob("wl*_run_*")):
    match = re.fullmatch(r"wl(\d+)_(frozen|joint)_run_(\d+)", folder.name)
    if match:
        rounds, mode, run_id = match.groups()
        label = f"{mode.capitalize()} CNN + WL (T={rounds}, run {run_id})"
        variant_dirs[label] = folder
class_names = {}
for label_file in (DATA_DIR / "labels").glob("*.json"):
    with label_file.open() as file:
        faces = json.load(file)["body"]["faces"]
    for face in faces:
        segment = face["segment"]
        class_names[segment["index"]] = segment["name"]
    if len(class_names) == 16:
        break

summary_rows, class_rows = [], []
for variant, directory in variant_dirs.items():
    # Final predictions are written only after training. Use the highest epoch
    # with both splits available, not a partially written final evaluation.
    completed = [folder for folder in directory.glob("epoch_*")
                 if folder.name.removeprefix("epoch_").isdigit()
                 and all((folder / f"{split}_predictions.pt").is_file()
                         for split in ("train", "validation"))]
    if not completed:
        print(f"No completed prediction files yet: {variant}")
        continue
    checkpoint_dir = max(completed, key=lambda folder: int(folder.name.split("_")[-1]))
    epoch = int(checkpoint_dir.name.split("_")[-1])
    for split in ("train", "validation"):
        results = torch.load(checkpoint_dir / f"{split}_predictions.pt",
                             map_location="cpu", weights_only=True)
        labels, predictions = results["labels"].long(), results["predictions"].long()
        num_classes = results["probabilities"].shape[1]
        confusion = torch.bincount(labels * num_classes + predictions,
                                  minlength=num_classes**2).reshape(num_classes, num_classes)
        intersection = confusion.diag()
        support = confusion.sum(1)
        union = support + confusion.sum(0) - intersection
        iou = intersection.double() / union.double().masked_fill(union == 0, float("nan"))
        per_class_accuracy = intersection.double() / support.double().masked_fill(support == 0, float("nan"))
        summary_rows.append(dict(variant=variant, epoch=epoch, split=split,
                                 faces=labels.numel(), loss=results["loss"],
                                 accuracy=intersection.sum().item() / labels.numel(),
                                 mean_iou=iou.nanmean().item()))
        for index in range(num_classes):
            class_rows.append(dict(variant=variant, epoch=epoch, split=split,
                                   class_index=index, class_name=class_names.get(index, str(index)),
                                   support=support[index].item(), iou=iou[index].item(),
                                   accuracy=per_class_accuracy[index].item()))

if summary_rows:
    summary = pd.DataFrame(summary_rows).set_index(["variant", "epoch", "split"])
    per_class_metrics = pd.DataFrame(class_rows)
    display(summary.style.format({"loss": "{:.4f}", "accuracy": "{:.2%}", "mean_iou": "{:.2%}"}))
    print("Per-class IoU (unweighted class mean above, including stock):")
    display(per_class_metrics.pivot(index=["class_index", "class_name"],
                                    columns=["variant", "split"], values="iou").style.format("{:.2%}"))
    summary.to_csv(project_dir / "results/wl_variant_metrics.csv")
    per_class_metrics.to_csv(project_dir / "results/wl_variant_per_class_metrics.csv", index=False)
else:
    print("Run training through final prediction export before comparing metrics.")


## Loss comparison: baseline + WL variants

Plot saved epoch-level training and validation losses for baseline and completed WL runs only. Different WL round counts and CNN modes remain separate; graphlet and Laplacian themes are excluded.


In [ ]:
# Comparison from saved histories; no inference or training.
import matplotlib.pyplot as plt
import pandas as pd

import re
variants = {"Geometry only": project_dir / "results/baseline_run_01/history.csv"}
for folder in sorted((project_dir / "results").glob("wl*_run_*")):
    match = re.fullmatch(r"wl(\d+)_(frozen|joint)_run_(\d+)", folder.name)
    if match:
        rounds, mode, run_id = match.groups()
        label = f"{mode.capitalize()} CNN + WL (T={rounds}, run {run_id})"
        variants[label] = folder / "history.csv"
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plotted = 0
for label, path in variants.items():
    if not path.is_file():
        print(f"Not run yet: {label}")
        continue
    frame = pd.read_csv(path)
    if frame.empty:
        continue
    for ax, metric in zip(axes, ("train_loss", "validation_loss")):
        ax.plot(frame["epoch"], frame[metric], marker="o", markersize=3, label=label)
    plotted += 1
for ax, title in zip(axes, ("Training loss", "Validation loss")):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean cross-entropy per face")
    ax.grid(alpha=0.3)
    if plotted:
        ax.legend()
fig.tight_layout()
if plotted:
    comparison_path = project_dir / "results/wl_variants_loss.png"
    fig.savefig(comparison_path, dpi=150)
    print("Saved comparison:", comparison_path)
plt.show()
